# Experiment 04: Implement SVM and KNN Classifiers in Python

**Aim:** To implement and evaluate Support Vector Machine (SVM) and k-Nearest Neighbors (kNN) classifiers on the Pima Indians Diabetes dataset, optimize k, compare linear and RBF kernels, and analyze diagnostic confusion matrices.

### Dataset Description
**Dataset:** Pima Indians Diabetes Diagnostic Dataset (768 female patients, 8 clinical diagnostic features).
- `preg`: Number of pregnancies (0-17).
- `glucose`: Plasma glucose concentration.
- `bp`: Diastolic blood pressure (mm Hg).
- `skin`: Triceps skin fold thickness (mm).
- `insulin`: 2-hour serum insulin (mu U/ml).
- `bmi`: Body Mass Index (weight in kg / height in m^2).
- `dpf`: Diabetes pedigree function.
- `age`: Age in years.
- `outcome`: Diagnostic status (0: Non-diabetic [500], 1: Diabetic [268]).

### Theory
kNN is a non-parametric, distance-based algorithm assigning class labels via majority vote among k closest neighbors using Euclidean metric. Support Vector Machines construct optimal hyperplanes that maximize the margin between classes: max 2/||w|| subject to margin separation. Kernel tricks, such as Radial Basis Function (RBF) K(x, x') = exp(-gamma * ||x - x'||^2), map features into infinite-dimensional Hilbert spaces to separate non-linearly separable classes. Both methods are sensitive to distance distortions, requiring strict standardization.

#### 1. Load Data, Treat Biologically Impossible Zeroes, and Stratified Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cols = ['preg', 'glucose', 'bp', 'skin', 'insulin', 'bmi', 'dpf', 'age', 'outcome']
df = pd.read_csv('../datasets/pima_diabetes.csv', header=None, names=cols)
X = df.drop(columns='outcome').copy()
y = df['outcome']
# Treat zero values in biological variables as missing
for col in ['glucose', 'bp', 'skin', 'insulin', 'bmi']:
    X[col] = X[col].replace(0, np.nan)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
for col in ['glucose', 'bp', 'skin', 'insulin', 'bmi']:
    med = Xtr[col].median()
    Xtr[col] = Xtr[col].fillna(med)
    Xte[col] = Xte[col].fillna(med)
print(f'Train set: {len(Xtr)}, Test set: {len(Xte)}')

#### 2. Optimize k in k-Nearest Neighbors

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15]
knn_scores = []
for k in k_values:
    pipe_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe_knn.fit(Xtr, ytr)
    knn_scores.append(pipe_knn.score(Xte, yte))
    print(f'kNN (k={k:2d}): Test Accuracy = {knn_scores[-1]:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(k_values, knn_scores, marker='o', color='navy')
plt.title('kNN Test Accuracy vs k Value')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Test Accuracy')
plt.grid(True)
plt.show()

#### 3. Support Vector Machine with Linear and RBF Kernels

In [ ]:
for kernel in ['linear', 'rbf']:
    svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, random_state=42))
    svm.fit(Xtr, ytr)
    print(f'SVM ({kernel.upper()} kernel): Test Accuracy = {svm.score(Xte, yte):.4f}')

#### 4. Best Model Confusion Matrix and Classification Diagnostics

In [ ]:
best_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
best_model.fit(Xtr, ytr)
pred = best_model.predict(Xte)
print('\nkNN (k=5) Confusion Matrix:\n', confusion_matrix(yte, pred))
print('\nClassification Report:\n', classification_report(yte, pred, target_names=['Non-Diabetic', 'Diabetic'], digits=4))

### Observations
1. Treating biological zero values as missing and imputing training medians significantly stabilizes distance calculations.
2. kNN accuracy peaked at k=5 with 75.32% test accuracy; smaller k (k=1: 72.08%) showed susceptibility to local noise.
3. The RBF SVM (74.03%) outperformed the Linear SVM (70.13%), confirming mild non-linear boundary characteristics in glucose-insulin-BMI interactions.
4. The kNN (k=5) confusion matrix correctly detected 34 of 54 diabetic cases (63.0% sensitivity) and 82 of 100 non-diabetic cases (82.0% specificity).

### Result
Feature standardization inside a leakage-proof pipeline proved critical for both distance-based classifiers. kNN with k=5 and RBF SVM provided comparable performance (~74-75% accuracy), with kNN offering marginally higher sensitivity for medical screening.